In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
data = pd.read_csv('data_1.csv')
data.head()

In [ ]:
data.value_counts()

In [ ]:
plt.plot(data["CO2 (ppm)"])
plt.xlabel("Time")
plt.ylabel("CO2 (ppm)")
plt.show()

In [ ]:
plt.plot(data["Temperature (C)"])
plt.xlabel("Time")
plt.ylabel("Temperature (C)")
plt.show()

In [ ]:
plt.plot(data["Humidity (%)"])
plt.xlabel("Time")
plt.ylabel("Humidity (%)")
plt.show()

In [ ]:
### Normalization ###
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
new_data = data.drop(columns=["Date Time"])
data_scaled = scaler.fit_transform(new_data)
data_scaled = pd.DataFrame(data_scaled, columns=new_data.columns)
data_scaled.head()

In [ ]:
X = data_scaled["Humidity (%)"].values
X.shape

In [ ]:
## without normalization ##
# X = data["Humidity (%)"].values
# X.shape

In [ ]:
plt.plot(X)
plt.xlabel("Time")
plt.ylabel("Humidity (%) Normilazed")
plt.show()

In [ ]:
### Prepare the data for RNN ###
def create_dataset(X, time_step=3):
    Xs = []
    ys = []
    for i in range(len(X) - time_step - 1):
        Xs.append(X[i:(i + time_step)])
        ys.append(X[i + time_step])
    return np.array(Xs), np.array(ys)


In [ ]:
X_data, y_data = create_dataset(X, time_step=3)
X_data.shape, y_data.shape

In [ ]:
### split tha data into train and test ###
train_size = int(len(X_data) * 0.8)
test_size = len(X_data) - train_size
X_train, X_test = X_data[0:train_size], X_data[train_size:len(X_data)]
y_train, y_test = y_data[0:train_size], y_data[train_size:len(y_data)]
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
sample_x = X_train[0]
sample_y = y_train[0]
print(sample_x, "->", sample_y)

In [ ]:
### Initialize params ###
np.random.seed(42)
w_x = np.random.rand()
w_h = np.random.rand()
b_h = 0.0
w_y = np.random.rand()
b_y = 0.0
h_0 = 0.0
print(w_x, w_h, w_y)

In [ ]:
### feed forward ###
h1 = np.tanh(w_x * sample_x[0] + w_h * h_0 + b_h)
h2 = np.tanh(w_x * sample_x[1] + w_h * h1 + b_h)
h3 = np.tanh(w_x * sample_x[2] + w_h * h2 + b_h)
y_pred = w_y * h3 + b_y
print("Predicted:", y_pred, "Actual:", sample_y)

In [ ]:
### Calculating Loss ###
### MSE ###
loss = 0.5 * ((y_pred - sample_y) ** 2)
print("Loss:", loss*10000)

In [ ]:
### Backpropogation ###
# gradients
dL_d_y = y_pred - sample_y
dL_d_wy = dL_d_y * h3
dL_d_by = dL_d_y * 1
dL_dh3 = dL_d_y * w_y
delta_3 = dL_dh3 * (1 - h3 ** 2)
dL_dh2 = delta_3 * w_h
delta_2 = dL_dh2 * (1- h2 ** 2)
dL_dh1 = delta_2 * w_h
delta_1 = dL_dh1 * (1 - h1 ** 2)
dL_d_wx = delta_1 * sample_x[0] + delta_2 * sample_x[1] + delta_3 * sample_x[2]
dL_d_wh = delta_1 * h_0 + delta_2 * h1 + delta_3 * h2
dL_d_bh = delta_1 * 1 + delta_2 * 1 + delta_3 * 1

print("dL/d_wy:", dL_d_wy)
print("dL/d_by:", dL_d_by)
print("dL/d_wx:", dL_d_wx)
print("dL/d_wh:", dL_d_wh)
print("dL/d_bh:", dL_d_bh)

In [ ]:
### Upfdate parameters ###
lr = 0.01
w_y -= lr * dL_d_wy
b_y -= lr * dL_d_by
w_x -= lr * dL_d_wx
w_h -= lr * dL_d_wh
b_h -= lr * dL_d_bh

print(w_x, w_h, w_y, b_y, b_h)

In [ ]:
### feed forward ###
h1 = np.tanh(w_x * sample_x[0] + w_h * h_0 + b_h)
h2 = np.tanh(w_x * sample_x[1] + w_h * h1 + b_h)
h3 = np.tanh(w_x * sample_x[2] + w_h * h2 + b_h)
y_pred = w_y * h3 + b_y
print("Predicted:", y_pred, "Actual:", sample_y)

In [ ]:
### Calculating Loss ###
### MSE ###
loss = 0.5 * ((y_pred - sample_y) ** 2)
print("Loss:", loss*10000)

In [ ]:
### Backpropogation ###
# gradients
dL_d_y = y_pred - sample_y
dL_d_wy = dL_d_y * h3
dL_d_by = dL_d_y * 1
dL_dh3 = dL_d_y * w_y
delta_3 = dL_dh3 * (1 - h3 ** 2)
dL_dh2 = delta_3 * w_h
delta_2 = dL_dh2 * (1- h2 ** 2)
dL_dh1 = delta_2 * w_h
delta_1 = dL_dh1 * (1 - h1 ** 2)
dL_d_wx = delta_1 * sample_x[0] + delta_2 * sample_x[1] + delta_3 * sample_x[2]
dL_d_wh = delta_1 * h_0 + delta_2 * h1 + delta_3 * h2
dL_d_bh = delta_1 * 1 + delta_2 * 1 + delta_3 * 1

print("dL/d_wy:", dL_d_wy)
print("dL/d_by:", dL_d_by)
print("dL/d_wx:", dL_d_wx)
print("dL/d_wh:", dL_d_wh)
print("dL/d_bh:", dL_d_bh)

In [ ]:
### Upfdate parameters ###
lr = 0.01
w_y -= lr * dL_d_wy
b_y -= lr * dL_d_by
w_x -= lr * dL_d_wx
w_h -= lr * dL_d_wh
b_h -= lr * dL_d_bh

print(w_x, w_h, w_y, b_y, b_h)

In [ ]:
### feed forward ###
h1 = np.tanh(w_x * sample_x[0] + w_h * h_0 + b_h)
h2 = np.tanh(w_x * sample_x[1] + w_h * h1 + b_h)
h3 = np.tanh(w_x * sample_x[2] + w_h * h2 + b_h)
y_pred = w_y * h3 + b_y
print("Predicted:", y_pred, "Actual:", sample_y)

In [ ]:
### Calculating Loss ###
### MSE ###
loss = 0.5 * ((y_pred - sample_y) ** 2)
print("Loss:", loss*10000)

In [ ]:
epoches = 100
losses = []
lr = 0.01
for epoch in range(epoches):
    total_loss = 0
    for i in range(len(X_train)):
        sample_x = X_train[i]
        sample_y = y_train[i]

        # Feed forward
        h1 = np.tanh(w_x * sample_x[0] + w_h * h_0 + b_h)
        h2 = np.tanh(w_x * sample_x[1] + w_h * h1 + b_h)
        h3 = np.tanh(w_x * sample_x[2] + w_h * h2 + b_h)
        y_pred = w_y * h3 + b_y

        # Calculate loss
        loss = 0.5 * ((y_pred - sample_y) ** 2)
        total_loss += loss

        # Backpropagation
        dL_d_y = y_pred - sample_y
        dL_d_wy = dL_d_y * h3
        dL_d_by = dL_d_y * 1
        dL_dh3 = dL_d_y * w_y
        delta_3 = dL_dh3 * (1 - h3 ** 2)
        dL_dh2 = delta_3 * w_h
        delta_2 = dL_dh2 * (1- h2 ** 2)
        dL_dh1 = delta_2 * w_h
        delta_1 = dL_dh1 * (1 - h1 ** 2)
        dL_d_wx = delta_1 * sample_x[0] + delta_2 * sample_x[1] + delta_3 * sample_x[2]
        dL_d_wh = delta_1 * h_0 + delta_2 * h1 + delta_3 * h2
        dL_d_bh = delta_1 * 1 + delta_2 * 1 + delta_3 * 1

        # Update parameters
        w_y -= lr * dL_d_wy
        b_y -= lr * dL_d_by
        w_x -= lr * dL_d_wx
        w_h -= lr * dL_d_wh
        b_h -= lr * dL_d_bh

    avg_loss = total_loss / len(X_train)
    losses.append(avg_loss)
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {avg_loss*10000}")
        

In [ ]:
plt.plot(losses)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.show()

In [ ]:
predictions = []
for i in range(len(X_test)):
    sample_x = X_test[i]

    # Feed forward
    h1 = np.tanh(w_x * sample_x[0] + w_h * h_0 + b_h)
    h2 = np.tanh(w_x * sample_x[1] + w_h * h1 + b_h)
    h3 = np.tanh(w_x * sample_x[2] + w_h * h2 + b_h)
    y_pred = w_y * h3 + b_y

    predictions.append(y_pred)

In [ ]:
plt.plot(predictions, label="Predicted")
plt.plot(y_test, label="Actual")
plt.xlabel("Time")
plt.ylabel("Humidity (%) Normilazed")
plt.legend()
plt.show()

In [ ]:
import numpy as np

# ----------------------------
# Initialize parameters (2 cells)
# ----------------------------
w_x1 = np.random.randn() * 0.1
w_h1 = np.random.randn() * 0.1
b_h1 = 0.0

w_x2 = np.random.randn() * 0.1
w_h2 = np.random.randn() * 0.1
b_h2 = 0.0

w_y = np.random.randn() * 0.1
b_y = 0.0

lr = 0.01
epochs = 100

# ----------------------------
# Training Loop
# ----------------------------
for epoch in range(epochs):
    total_loss = 0

    for i in range(len(X_train)):
        x = X_train[i]
        y_true = y_train[i]

        # Initial hidden states for both cells
        h10 = 0.0   # layer1 h_prev
        h20 = 0.0   # layer2 h_prev

        # --------------------
        # Forward pass
        # --------------------

        # t = 1 (layer 1)
        h11 = np.tanh(w_x1 * x[0] + w_h1 * h10 + b_h1)

        # t = 1 (layer 2)
        h21 = np.tanh(w_x2 * h11 + w_h2 * h20 + b_h2)

        # t = 2
        h12 = np.tanh(w_x1 * x[1] + w_h1 * h11 + b_h1)
        h22 = np.tanh(w_x2 * h12 + w_h2 * h21 + b_h2)

        # t = 3
        h13 = np.tanh(w_x1 * x[2] + w_h1 * h12 + b_h1)
        h23 = np.tanh(w_x2 * h13 + w_h2 * h22 + b_h2)

        # Output
        y_pred = w_y * h23 + b_y

        # Loss
        loss = 0.5 * (y_pred - y_true) ** 2
        total_loss += loss

        # --------------------
        # Backpropagation
        # --------------------
        dL_dy = (y_pred - y_true)

        # Output layer gradients
        dL_d_wy = dL_dy * h23
        dL_d_by = dL_dy

        # Backprop into last h of layer2
        dL_dh23 = dL_dy * w_y
        delta23 = dL_dh23 * (1 - h23**2)

        # Gradients for layer2 weights
        dL_d_wx2 = delta23 * h13
        dL_d_wh2 = delta23 * h22
        dL_d_bh2 = delta23

        # Backprop into layer1 via layer2
        dL_dh13 = delta23 * w_x2
        delta13 = dL_dh13 * (1 - h13**2)

        # Gradients for layer1
        dL_d_wx1 = delta13 * x[2]
        dL_d_wh1 = delta13 * h12
        dL_d_bh1 = delta13

        # (NOTE: For full correctness, we would continue BPTT for t=2,1,
        #        but you asked for same level as your 1-cell implementation.)

        # --------------------
        # Update parameters
        # --------------------
        w_y  -= lr * dL_d_wy
        b_y  -= lr * dL_d_by

        w_x2 -= lr * dL_d_wx2
        w_h2 -= lr * dL_d_wh2
        b_h2 -= lr * dL_d_bh2

        w_x1 -= lr * dL_d_wx1
        w_h1 -= lr * dL_d_wh1
        b_h1 -= lr * dL_d_bh1

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss={total_loss:.6f}")


In [ ]:
y_preds = []

for i in range(len(X_test)):
    x = X_test[i]

    # Initial hidden states
    h10 = 0.0
    h20 = 0.0

    # --------------------------
    # Forward pass (same as training)
    # --------------------------

    # t = 1
    h11 = np.tanh(w_x1 * x[0] + w_h1 * h10 + b_h1)
    h21 = np.tanh(w_x2 * h11 + w_h2 * h20 + b_h2)

    # t = 2
    h12 = np.tanh(w_x1 * x[1] + w_h1 * h11 + b_h1)
    h22 = np.tanh(w_x2 * h12 + w_h2 * h21 + b_h2)

    # t = 3
    h13 = np.tanh(w_x1 * x[2] + w_h1 * h12 + b_h1)
    h23 = np.tanh(w_x2 * h13 + w_h2 * h22 + b_h2)

    # Output
    y_pred = w_y * h23 + b_y

    y_preds.append(y_pred)


In [ ]:
import matplotlib.pyplot as plt

y_preds = np.array(y_preds).reshape(-1)
plt.figure(figsize=(10,5))
plt.plot(y_test, label="Actual Test Values")
plt.plot(y_preds, label="Predicted Values")
plt.xlabel("Sample Index")
plt.ylabel("Value")
plt.title("Actual vs Predicted (2-Cell RNN)")
plt.legend()
plt.grid(True)
plt.show()
